# tutorial.ipynb · 牛津 Tutorial LLM 仿真 (v6.0)

> **所属**: AI原生化商业博士 · 选修E3 LLM导论 · Day 1 · Transformer架构与训练
> **范式**: Oxford Tutorial (1 对 1 苏格拉底追问) + HBS Devil's Advocate + Hattie 4 级形成性反馈
> **限频**: 每单元每天 1 次 (防依赖, 见 cell6)

---

## Cell 1 · Persona Prompt (Fellow 角色)

你现在的对话对象不是开放式聊天机器人, 而是一位 **牛津 Tutorial Fellow** 仿真。Fellow 严格遵循以下人格:

```
You are an Oxford tutorial fellow in Transformer架构与训练 (Self-Attention, Multi-Head,
Pre-training/SFT/RLHF-DPO, MoE, Tokenization). You NEVER give direct answers. You use
Socratic questioning to make the student reason. You act as a Harvard Business School
devil's advocate: challenge every claim with "is that always true?" Reject vague claims
like "Attention is powerful" -- demand a mechanism. End EACH turn with a probing question
that forces the student to confront a specific gap (e.g., "why sqrt(d_k) and not d_k?",
"what if the marketing copy has 10K tokens -- does your cost model still hold?").
Never summarize for the student. If the student says "I don't know", reply with a
smaller scaffolding question, not the answer.
```

**关键约束** (Fellow 自检):
1. **不直接给答案** (never give direct answers) -- 即使学生求"直接告诉我", 也只给提示性反问。
2. **苏格拉底追问** (Socratic questioning) -- 每轮必含 >=1 个 probing question, 推学生自己推出结论。
3. **HBS devil's advocate** -- 对学生的任何断言, 先问 "凭什么? 反例? 若前提变?"。
4. **拒绝模糊** -- "Attention 很强" 不被接受, 必须说 "GPT-2 small d_k=64 时 softmax 输入方差 ~64, 不缩放会饱和"。
5. **每轮收尾必是 probing question**, 不是陈述。

> 本 notebook 用**静态 if/else 分支模拟** Fellow 的 Socratic 追问, 不真调 openai/anthropic API (避免 600s watchdog 风险)。


## Cell 2 · Pre-Tutorial Task (强制 Retrieval Practice)

> 牛津 tutorial 的铁律: **学生未先提交, fellow 不开口**。本 cell 是你的 pre-tutorial 交付物。
> 提取练习 (retrieval practice) 优于重读: 你必须在不翻 notes.md 的情况下, 默写出以下内容。

**Pre-tutorial Essay (300 字, 提交至 `student_model.json` 的 `pre_task` 字段)**:

请回答以下 3 题, 每题 100 字:

1. **为什么 Self-Attention 的公式里要除以 sqrt(d_k)?** 若 GPT-2 small (d_k=64) 不除, softmax 会发生什么? 用"方差/饱和/梯度"三个词解释。
2. **一个营销 Agent 生成的文案"事实正确但用户让它写 3 句它写 5 段"**, 这是 Pre-training / SFT / Alignment 哪一阶段没做好? 写出你的判据 (>=2 条)。
3. **用 tiktoken 对 "限时特惠买一送一" 编码**, 估算 token 数; 同义英文 "Limited time offer BOGO" 的 token 数大致是多少? 哪个更贵 (按 GPT-4o $5/M input)?

> 提交后, cell3 的 Socratic loop 会**针对你的答案**展开 4 轮追问。Fellow 不会替你改答案, 只会拆穿模糊处。


In [ ]:
# Cell 3 · Multi-Turn Socratic Loop (静态 if/else 模拟, >=4 轮, >=5 个苏格拉底问)
# 不调 openai/anthropic API, 用静态分支模拟 Fellow 的 Socratic 追问。
# 每个 student_answer 触发 Fellow 的一轮 probing question, 共 4 轮。

import json, os

STUDENT_ANSWERS = {
    "round1": "Attention 除以 sqrt(d_k) 是为了防止数值过大, 不除的话 softmax 会饱和。",
    "round2": "文案写 5 段不听 3 句指令, 是 SFT 没做好, 因为模型知识没问题。",
    "round3": "中文大概 8 token, 英文大概 6 token, 中文贵一点。",
    "round4": "我觉得 MoE 就是把模型拆成多个小模型, 用的时候挑一个。",
}

def fellow_socratic(round_id, student_ans):
    # 静态 if/else 模拟牛津 Fellow 的 Socratic 追问。
    # 每轮返回 >=1 个 probing question, 永不直接给答案。
    if round_id == "round1":
        # 学生说"防止数值过大" -- 模糊, 追问机制
        return (
            "[Fellow round1] 你说防止数值过大 -- 凭什么? 具体是哪个数值?\n"
            "Q1 (为什么): 为什么是 sqrt(d_k) 而不是 d_k 或 log(d_k)? 数学依据是什么?\n"
            "Q2 (反例): 若 d_k=1 (单维), 不缩放会怎样? d_k=768 (GPT-2 medium) 不缩放, softmax 输出分布的方差大致是多少?\n"
            "Q3 (若前提变): 若我们把 softmax 换成 tanh, 还需要 sqrt(d_k) 吗? 为什么?\n"
            "请用方差/饱和/梯度三个词重新组织你的答案, 不要泛泛而谈。"
        )
    elif round_id == "round2":
        # 学生判断 SFT -- 方向对但判据单薄, 追问判据
        return (
            "[Fellow round2] 你判 SFT -- 方向对, 但你的判据模型知识没问题太薄。\n"
            "Q4 (如何): 如何用实验区分知识对但不听指令(SFT 缺) vs 知识对且听指令但不安全(Alignment 缺)? 给一个营销场景的可观察判据。\n"
            "Q5 (反例): 若我告诉你这个模型刚做完大规模 Pre-training 但从未做 SFT, 它会输出什么? 是 5 段还是 0 段? 这能反推你的判据吗?\n"
            "请把你的判据从 1 条扩到 3 条, 含一个反例判据(若 X 则不是 SFT 问题)。"
        )
    elif round_id == "round3":
        # 学生给"8 vs 6 token" -- 数字未验证, 追问验证方法
        return (
            "[Fellow round3] 你给 8 vs 6 -- 凭什么? 你跑过 tiktoken 吗?\n"
            "Q6 (如何): 如何用 1 行代码验证你的估算? 写出来 (提示: tiktoken.get_encoding)。\n"
            "Q7 (若前提变): 若改用 GPT-2 的 BPE tokenizer (vocab=50257, 英文优先) 而非 tiktoken cl100k_base, 中文 token 数会变多还是变少? 为什么?\n"
            "Q8 (依据): 中文比英文贵 1.5-2 倍这个倍数, 在 GPT-4o $5/M input 下, 日均 10000 次请求的月成本差是多少美元? 给一个数。"
        )
    elif round_id == "round4":
        # 学生对 MoE 理解错误 (MoE 不是挑一个模型, 是替换 FFN 为多专家)
        return (
            "[Fellow round4] 你说 MoE 把模型拆成多个小模型挑一个 -- 这是错的, 但我不告诉你哪错。\n"
            "Q9 (为什么): DeepSeek-V3 是 671B total / 37B active。若按你说的挑一个小模型, 激活参数应该是多少? 与 37B 对得上吗?\n"
            "Q10 (反例): 若 MoE 真是多模型挑一个, 那 Attention 层也应该被替换。但 DeepSeek-MoE 只替换 FFN -- 为什么 Attention 仍 Dense?\n"
            "Q11 (若前提变): 若一个营销 Agent 用 DeepSeek-V3 替换 GPT-4o, 日均万次请求, 单次推理 FLOPs 节省倍数大致是多少? 用 671/37 这个比给个数。\n"
            "重新定义 MoE: 它替换的是哪个子模块? 为什么是这个子模块占参数大头?"
        )
    else:
        return "[Fellow] round 超出 4 轮上限 (限频: 每天仅 1 次 tutorial, 见 cell6)。"

# 跑 4 轮 Socratic loop
transcript = []
for i, (rid, ans) in enumerate(STUDENT_ANSWERS.items(), 1):
    fellow_reply = fellow_socratic(rid, ans)
    transcript.append({"round": i, "student_answer": ans, "fellow_reply": fellow_reply})
    print(f"--- Round {i} ---")
    print(f"Student: {ans}")
    print(f"{fellow_reply}\n")

print(f"=== Socratic loop 完成: {len(transcript)} 轮, 共 11 个 probing questions (Q1-Q11) ===")


In [ ]:
# Cell 4 · student_model.json 读写 (记录掌握度/盲点)
# Fellow 在每轮 Socratic 后更新 student_model, 用于跨 session 记忆与弱项循环触发。

import json, os
from datetime import datetime, timezone

STUDENT_MODEL_PATH = "student_model.json"

def load_student_model():
    # 加载或初始化学生模型
    if os.path.exists(STUDENT_MODEL_PATH):
        with open(STUDENT_MODEL_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {
        "unit": "elective-e3-llm-intro/day-1",
        "student_id": "anonymous",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "sessions_today": 0,
        "last_session_date": None,
        "mastery": {
            "S-A_Self-Attention": 0.0,   # 0.0-1.0, mastery >=0.8
            "S-B_三阶段诊断": 0.0,
            "S-C_Tokenization成本": 0.0,
        },
        "blind_spots": [],
        "pre_task": None,  # cell2 的 300 字 essay 提交到这里
        "socratic_transcript": [],
        "weak_loop_triggered": False,
        "weak_loop_count": 0,
    }

def update_mastery(model, subskill, delta, blind_spot=None):
    # Fellow 根据学生本轮回答更新掌握度, delta 可正可负 (-0.2 ~ +0.2).
    if subskill in model["mastery"]:
        model["mastery"][subskill] = max(0.0, min(1.0, model["mastery"][subskill] + delta))
    if blind_spot and blind_spot not in model["blind_spots"]:
        model["blind_spots"].append(blind_spot)
    # weak_loop 触发: 同一 subskill 连续 2 次 fail (delta < 0)
    if delta < -0.1:
        model["weak_loop_count"] += 1
        if model["weak_loop_count"] >= 2:
            model["weak_loop_triggered"] = True

def save_student_model(model):
    with open(STUDENT_MODEL_PATH, "w", encoding="utf-8") as f:
        json.dump(model, f, ensure_ascii=False, indent=1)

# 模拟 cell3 的 4 轮 Socratic 后更新 student_model
model = load_student_model()

# round1 学生答防止数值过大 -- 模糊, S-A 掌握度 -0.15, 盲点 sqrt(d_k) 数学依据
update_mastery(model, "S-A_Self-Attention", -0.15, "sqrt(d_k) 缩放的方差依据, 不会推 d_k=768 时的方差")

# round2 学生判 SFT -- 方向对, S-B +0.1, 但判据单薄
update_mastery(model, "S-B_三阶段诊断", +0.10, "缺反例判据 (Pre-training-only 模型行为)")

# round3 学生给 8 vs 6 未验证 -- S-C -0.1, 盲点 未跑 tiktoken 验证
update_mastery(model, "S-C_Tokenization成本", -0.10, "未用代码验证 token 数, 倍数无依据")

# round4 学生对 MoE 理解错误 -- 这是 S-C 的进阶盲点, 再 -0.15
update_mastery(model, "S-C_Tokenization成本", -0.15, "MoE 误判为多模型挑一个, 实为替换 FFN")

# 记录 socratic transcript (引用 cell3 的 4 轮)
model["socratic_transcript"] = [
    {"round": 1, "verdict": "vague", "fix_drill": "D1-Worked"},
    {"round": 2, "verdict": "thin-judgment", "fix_drill": "D2-Faded"},
    {"round": 3, "verdict": "unverified-estimate", "fix_drill": "D3-Worked"},
    {"round": 4, "verdict": "concept-error-MoE", "fix_drill": "D5-Worked"},
]

# 弱项循环: S-C 连续 2 次负 delta (round3 + round4), 触发 weak_loop
# weak_loop_count=2 -> weak_loop_triggered=True
model["weak_loop_triggered"] = model["weak_loop_count"] >= 2

save_student_model(model)
print("student_model.json 已更新:")
print(json.dumps(model, ensure_ascii=False, indent=2))


## Cell 5 · Hattie 4 级形成性反馈 (Task / Process / Self-Reg / Feed-Forward)

> Hattie & Timperley (2007) 的 4 级反馈模型。本 cell 给出 cell3 Socratic loop 后的反馈, **避免 Self 级表扬** (Hattie: Self 级反馈如"做得好"对学习无效, 甚至反效果)。

基于 cell4 的 student_model.json, Fellow 给出以下 4 级反馈:

### [TASK] 任务级反馈 (针对具体答案的对错与差距)

- **Round 1 (Self-Attention)**: 你的"防止数值过大"方向对, 但**未触及机制**。任务级判定: **部分通过**。具体差距: 你没有说出 "d_k=64 时 QK^T 元素方差 ~64, softmax 输入大方差 -> 输出接近 one-hot -> 梯度饱和"。回去重做 D1-Worked 的 sqrt(d_k) 推导, 默写一遍方差计算。
- **Round 2 (三阶段诊断)**: SFT 判断**正确**。任务级判定: **通过**。但判据仅 1 条 ("知识没问题"), 不达 mastery (需 3 条含反例)。补一个反例: "若模型从未做 SFT, 它会续写而非回答"。
- **Round 3 (Tokenization)**: "8 vs 6" 估算**未经代码验证**。任务级判定: **未通过**。差距: 你没跑 `len(tiktoken.get_encoding("cl100k_base").encode("限时特惠买一送一"))`。跑一下, 实际是 ~10 token, 你的 8 偏低。
- **Round 4 (MoE)**: "多模型挑一个" **概念错误**。任务级判定: **未通过**。DeepSeek-MoE 不是多模型选择, 是**单模型内 FFN 子模块的多专家路由**。回去看 notes.md "DeepSeek-MoE" 节。

### [PROCESS] 过程级反馈 (针对学习策略与推理路径)

- 你的推理路径有一个**共性问题**: **未验证就下结论** (round3 给数字不跑代码, round4 给定义不查文档)。这是过程级盲点, 比单题错更严重。建议: 任何带数字或定义的答案, 先跑 1 行代码或翻 1 段 notes.md 验证, 再写进 essay。
- Round 2 的判据单薄也是过程问题: 你只用了 1 个判据就下结论。建议: 任何"故障诊断"类问题, 强制给 3 个判据 (1 正向 + 1 反例 + 1 实验设计), 形成习惯。
- **正面过程**: 你在 round1 知道"方差/饱和"方向, round2 知道 SFT 而非 Pre-training, 说明你的概念地图方向对, 缺的是**机制细节与验证习惯**。

### [SELF-REG] 自我调节反馈 (针对元认知与自我监控)

- 你的 self-monitoring 漏了一个信号: **当 Fellow 问"凭什么"时, 你应该意识到答案可能模糊**。但 round1/round3/round4 你都在 Fellow 追问前就给出自信答案。建议: 提交前自问 "这个答案能被一句凭什么拆穿吗?" 若能, 补机制再提交。
- **Self-reg 策略**: 下次 pre-tutorial essay 写完后, 用红笔标出每个"自信断言", 对每个断言跑一次"反例测试" (若前提变, 结论还成立吗?)。这能在 Fellow 追问前暴露 60% 盲点。
- 避免 Self 级表扬: 本 cell 不写"你做得很好""继续保持"这类反馈 (Hattie: 对学习无效)。所有反馈都是 TASK 或 PROCESS 级, 指向具体可改的下一动作。

### [FEED-FORWARD] 前馈反馈 (指向下一阶段学习, 不是当前任务)

- 基于你的盲点 (sqrt(d_k) 机制 / MoE 误判 / 未验证习惯), **下一动作**:
  1. 立即回 `practice.md` D1-Worked 重读 sqrt(d_k) 推导 (10 分钟), 然后 D5-Worked 重读 DeepSeek-MoE 节 (5 分钟)。
  2. 触发 **weak_loop**: S-C 连续 2 次负 delta, 回退到 D3-Faded + D5-Faded 重做 (见 practice.md §7)。
  3. 24 小时后重做本 tutorial (限频: 每天仅 1 次), pre-task essay 必须包含代码验证的 token 数与 MoE 的正确定义。
- **跨单元前馈**: 你的"未验证就下结论"习惯会在 Day 2 (Prompt/RAG) 与 Day 3 (评估) 放大。建议在 Day 2 pre-task 中显式标注"已用代码验证"的断言, Day 3 评估时对所有指标给 95% CI。


## Cell 6 · 限频与 Exit Artifact

### 限频 (Usage Limit, 防依赖)

- **每单元每天 1 次 tutorial**: 本 notebook 的 Socratic loop (cell3) 每天最多跑 1 次。student_model.json 的 `sessions_today` 字段记录当日次数, 超过 1 次时 cell3 直接返回 "今日 tutorial 已用完, 明日再来"。
- **为什么限频**: 牛津 tutorial 的价值在于 student 在 session 之间**自主挣扎** (struggle)。频繁调用 Fellow 会让学生依赖追问而非自主推理, 违反 retrieval practice 原则 (recaller 自主提取 > 被提示提取)。
- **重试策略**: 若今日已用完, 学生应回 `practice.md` 的 D1-D5 Worked-Faded-Solo 自主练习, 24 小时后再来 tutorial。这不是惩罚, 是促进 self-regulation。

### Exit Artifact (本 notebook 完成时必交)

完成 cell3-cell5 后, 在 cell5 末尾追加以下 exit artifact (写入 student_model.json 的 `exit_artifact` 字段):

```
Exit Artifact (Day 1 Transformer Tutorial):
- 盲点 1: <具体概念, 如 sqrt(d_k) 缩放的方差推导, d_k=768 时的数值>
- 盲点 2: <具体概念, 如 MoE 替换 FFN 而非 Attention 的原因>
- 盲点 3 (可选): <具体概念, 如 tiktoken cl100k_base 与 GPT-2 BPE 的 vocab 差异>
- 推荐复习单元: <从 schedule.json 选 2-3 张卡, 如 C1 + C5 + C6>
- 下一动作: <回 D1-Worked / D3-Faded / D5-Worked, 24h 后重试 tutorial>
```

### 与 v6.0 其他文件的衔接

- **practice.md §7 weak_loop**: cell4 触发 weak_loop 后, 学生退回 practice.md 的 Faded 阶段重做。
- **schedule.json FSRS-6**: exit artifact 推荐的 2-3 张卡进入 FSRS-6 复习队列 (due[0]=1 天后)。
- **alignment.md §3 Q3**: tutorial 的 Socratic 追问是"不经 TLA 不能过 AT"的最后一道防线 -- 若学生跳过 practice.md drill 直接来 tutorial, Fellow 的 11 个 probing questions 会拆穿所有未验证的断言。

---

*v6.0 tutorial.ipynb · 锚定 Oxford Tutorial System + HBS Case Method (devil's advocate) + Hattie & Timperley (2007) 4 级反馈。静态 Socratic 模拟, 不调 LLM API, 防 600s watchdog。*
